# Optimised Artificial Bee Colony for Influence Maximisation on Signed Networks

This notebook is a guided walkthrough of the ABC implementation used in the
report. It contains the complete algorithm, cell by cell, with the same
parameters and operators that produced the measured results.

**What is covered**

1. The problem and the search objective.
2. The interface an ABC needs from a diffusion model.
3. The configuration object and every parameter.
4. The CELF warm start with an audited lazy bound.
5. The four move operators (initialisation, partner swap, exploration, tournament).
6. The main loop: employed bees, onlooker bees, scouts, dedupe, elitism.
7. A run on the nine-node demonstration graph from the report.
8. The same code on Bitcoin Alpha.
9. An equivalence check against the project module `signed_bench/abc_proposed.py`.

**How to run.** Start Jupyter from the repository root (the folder that contains
`signed_bench/`), then run all cells. The first demos need only `numpy`; the
Bitcoin Alpha demo reads the cached graph in `signed_bench/data/` and takes a
few seconds.

The canonical source is `signed_bench/abc_proposed.py`; the cells below reproduce
its logic with expanded comments. The diffusion model is documented in
`signed_bench/opinion_model.py` and in Section III of `REPORT.pdf`.

## 1. The problem

We are given a signed graph $G=(V,E)$ with edge signs and weights. A seed set
$S\subseteq V$ of size $k$ is planted at $+1$, and a two-opinion majority rule
propagates opinions:

$$c_{u\to v}=\operatorname{sign}(e_{uv})\,s_u\,w_{uv},\qquad
S_v=\sum_{u\in N(v)} c_{u\to v},\qquad
s_v=\operatorname{sign}(S_v)\ \text{if}\ |S_v|>\theta_v\deg(v).$$

The search objective is the expected number of positive-active nodes,

$$\sigma^{+}(S)=\mathbb{E}\big[\#\{v:s_v=+1\}\big],$$

plus a plateau term $\lambda\Phi(S)$ computed in the same cascade pass. The
budget is $k\times|P|$ cascade evaluations over a candidate pool $P$, the exact
cost of forward greedy over the pool.

The difficulty is that $\sigma^{+}$ is flat over large regions of the search
space: many seed sets give exactly the same cascade, so a method that only
compares final spread has no gradient. ABC searches whole $k$-sets and uses
$\Phi$ to break those ties.

In [1]:
import heapq
import sys
import time
from dataclasses import dataclass

import numpy as np

# The demos in Sections 8 and 9 import the project modules from this folder.
sys.path.insert(0, "signed_bench")

rng_preview = np.random.default_rng(0)
print("numpy", np.__version__)

numpy 2.5.4


## 2. What ABC needs from a model

The search never touches the graph directly. It only calls four members of a
model object:

| member | meaning |
|---|---|
| `model.sigma(S)` | expected positive spread of the seed set `S` |
| `model.sigma_potential(S)` | `(sigma, Phi, standard_error)` from one cascade pass |
| `model.n_evals` | number of cascade evaluations spent so far (budget counter) |
| `model.pool_weight` | per-node weight used by the biased initialiser |

Below is a small, deterministic implementation of the two-opinion rule for the
nine-node graph used in the report. It follows the same rule as the production
model in `signed_bench/opinion_model.py` (one draw instead of thirty, which is
enough to demonstrate the search).

In [2]:
class ToyOpinionModel:
    """Two-opinion signed majority rule on a small graph (one draw).

    States are -1 (negative-active), 0 (inactive), +1 (positive-active).
    A node activates when |support| >= floor(theta * deg) + 1, which is the
    integer form of the strict inequality in the report.
    """

    def __init__(self, edges, n, theta=0.5):
        self.n = n
        self.adj = [[] for _ in range(n + 1)]
        for u, v, sign in edges:
            self.adj[u].append((v, sign))
            self.adj[v].append((u, sign))

        # deg and pool_weight are the same quantity here: total incident weight
        self.deg = np.zeros(n + 1)
        for v in range(1, n + 1):
            self.deg[v] = len(self.adj[v])
        self.pool_weight = self.deg.copy()

        # integer activation threshold per node
        self.need = np.array([int(np.floor(theta * d)) + 1 if d else 10**9
                              for d in self.deg])

        self.n_evals = 0          # budget counter read by the search
        self.cache = {}           # memoised sigma values
        self.cache_pot = {}       # memoised (sigma, Phi, sem) triples

    def _run(self, S):
        """Run the cascade to convergence; return (state, final support)."""
        state = np.zeros(self.n + 1, dtype=int)
        for v in S:
            state[v] = 1
        support = np.zeros(self.n + 1, dtype=int)
        while True:
            support[:] = 0
            for u in range(1, self.n + 1):
                if state[u] != 0:
                    for v, sign in self.adj[u]:
                        support[v] += sign * state[u]
            new = (np.abs(support) >= self.need) & (state == 0)
            if not new.any():
                break
            state[new] = np.sign(support[new]).astype(int)
        return state, support

    def sigma(self, S):
        key = frozenset(S)
        if key not in self.cache:
            state, _ = self._run(S)
            self.cache[key] = float((state == 1).sum())
            self.n_evals += 1
        return self.cache[key]

    def sigma_potential(self, S):
        key = frozenset(S)
        if key not in self.cache_pot:
            state, support = self._run(S)
            phi = float(np.sum(np.clip(support / self.need, 0, 1)))
            self.cache_pot[key] = (self.sigma(S), phi, 0.0)
        return self.cache_pot[key]

In [3]:
# The nine-node demonstration graph of the report: 15 edges, 5 negative.
EDGES = [
    (3, 5, 1), (3, 7, 1), (6, 8, 1), (3, 8, 1), (5, 8, 1),
    (1, 9, 1), (3, 4, 1), (2, 8, 1), (4, 5, 1), (6, 7, 1),
    (2, 9, -1), (1, 8, -1), (1, 7, -1), (1, 3, -1), (1, 6, -1),
]
N = 9
toy = ToyOpinionModel(EDGES, N)

print("node  degree  need")
for v in range(1, N + 1):
    print(f"{v:4d} {int(toy.deg[v]):7d} {int(toy.need[v]):5d}")

# sanity check: the trace quoted in the report
state, _ = toy._run([3, 6])
print("\nseeds {3,6} ->", state[1:].tolist(),
      "| pos", int((state == 1).sum()), "neg", int((state == -1).sum()))

node  degree  need
   1       5     3
   2       2     2
   3       5     3
   4       2     2
   5       3     2
   6       3     2
   7       3     2
   8       5     3
   9       2     2

seeds {3,6} -> [-1, 0, 1, 1, 1, 1, 1, 1, 0] | pos 6 neg 1


## 3. Configuration

Every knob lives in one dataclass, so an experiment is fully described by its
`ABCConfig`. The values below are the defaults used in the reported runs. The
comments name the weakness each field addresses (W1-W9 in the report).

In [4]:
@dataclass
class ABCConfig:
    # colony ------------------------------------------------------------
    colony_size: int = 30        # 15 sources, each with an employed bee
    max_cycles: int = 80         # hard stop on generations
    limit: int = 6               # trials before a source is abandoned (scout)

    # move operators (W1-W5) -------------------------------------------
    mutation_rate: float = 0.2   # W4: probability of an extra random swap
    n_mod: int = 1               # W4: partner-guided swaps per candidate
    gbest_prob: float = 0.6      # W1: partner is the global best
    selection: str = "tournament"  # W2: "tournament" or "roulette"
    tournament_size: int = 4     # W2: candidates per onlooker tournament
    directed_scout: bool = True  # W3: rebuild near the incumbent
    dedupe: bool = True          # W5: no duplicate sources

    # plateau handling (W7) --------------------------------------------
    accept_equal: bool = True    # accept exact ties
    explore_prob: float = 0.15   # probability of a fresh random set
    potential_weight: float = 0.05   # lambda in fitness = sigma + lambda*Phi
    tol_mult: float = 2.0        # ties = differences within 2 standard errors

    # refinement (W6, W8, W9) ------------------------------------------
    refine_every: int = 0        # 0 disables refinement (used in all runs)
    reserve_fraction: float = 0.15   # budget share reserved for refinement

    # budget and seeding -----------------------------------------------
    seed_with_celf: bool = True  # warm start from audited CELF
    eval_budget: int | None = None   # hard cap on cascade evaluations
    random_state: int = 0        # reproducible restarts

## 4. Warm start: CELF with an audited lazy bound

CELF is a lazy greedy: it keeps a priority queue of stale marginal gains and
only re-evaluates a candidate when its bound is out of date. Under the signed
majority rule the bound is not valid, so the implementation **counts
violations** instead of assuming them away. The set it returns becomes the
first food source, which guarantees that a restart can never finish below
CELF.

In [5]:
def celf(model, k, pool):
    """Lazy greedy warm start. Returns a list of k seed nodes.

    The stale bound is a heuristic here (the objective is not submodular);
    violations are counted in the full implementation and reported in the CSV.
    """
    base = model.sigma([])                    # empty-set spread
    heap, stale = [], {}
    for v in pool:                            # first round: singletons
        gain = model.sigma([v]) - base
        stale[v] = gain
        heapq.heappush(heap, (-gain, v, 0))
    S, cur = [], base
    while len(S) < k and heap:
        neg_gain, v, last_round = heapq.heappop(heap)
        if last_round == len(S):              # bound is current: accept
            S.append(v)
            cur += -neg_gain
            continue
        new_gain = model.sigma(S + [v]) - cur  # stale bound: re-evaluate
        stale[v] = new_gain
        heapq.heappush(heap, (-new_gain, v, len(S)))
    return S

## 5. Move operators

Four small functions generate every candidate the colony evaluates.

- `_biased_set` samples a fresh $k$-set with probability proportional to node
  weight (used for initialisation and duplicate replacement).
- `_neighbour_solution` performs the partner-guided swap: it exchanges nodes
  that appear in the bee's set but not the partner's with nodes that appear in
  the partner's set but not the bee's (W1/W4), then optionally applies one
  purely random swap (the modification rate).
- `_candidate` occasionally returns a fresh random set instead (W7 exploration).
- `_pick_sources` implements the onlooker selection: a tournament of size 4 by
  default, with roulette kept as an option (W2).

In [6]:
def _biased_set(model, pool, k, rng):
    """Weight-biased random k-set (no duplicates)."""
    w = np.asarray(model.pool_weight + 1.0, dtype=float)
    w = w[np.asarray(pool, dtype=int)]
    w = w / w.sum()
    return rng.choice(pool, size=k, replace=False, p=w).tolist()


def _neighbour_solution(x, partner, pool, k, rng, cfg):
    """Partner-guided swaps plus an optional random mutation (W1, W4)."""
    partner_set = set(partner)
    xs = list(dict.fromkeys(x))
    for _ in range(max(1, cfg.n_mod)):
        only_i = [v for v in xs if v not in partner_set]
        only_j = [v for v in partner_set if v not in xs]
        if only_i and only_j:                  # inherit from the partner
            drop = only_i[rng.integers(len(only_i))]
            take = only_j[rng.integers(len(only_j))]
            xs[xs.index(drop)] = take
        else:                                  # no usable difference: random
            drop = xs[rng.integers(len(xs))]
            options = [v for v in pool if v not in xs]
            if options:
                xs[xs.index(drop)] = options[rng.integers(len(options))]
        if rng.random() < cfg.mutation_rate:   # extra random swap
            cur = set(xs)
            options = [v for v in pool if v not in cur]
            if options:
                drop = xs[rng.integers(len(xs))]
                xs[xs.index(drop)] = options[rng.integers(len(options))]
    return list(dict.fromkeys(xs))[:k]


def _candidate(x, partner, pool, k, rng, cfg):
    """Exploration move with probability explore_prob (W7)."""
    if rng.random() < cfg.explore_prob:
        return rng.choice(pool, size=k, replace=False).tolist()
    return _neighbour_solution(x, partner, pool, k, rng, cfg)


def _pick_sources(fitness, rng, cfg):
    """Tournament selection of onlooker sources (W2)."""
    n = len(fitness)
    fit = np.asarray(fitness, dtype=float)
    if cfg.selection == "roulette":
        shifted = fit - fit.min() + 1e-6
        probs = shifted / shifted.sum()
        return rng.choice(n, size=n, p=probs)
    picks = rng.integers(n, size=(n, max(2, cfg.tournament_size)))
    return picks[np.arange(n), np.argmax(fit[picks], axis=1)]

## 6. The main loop

One cycle runs five phases, in this order:

1. **Employed bees.** Every source produces one candidate. With probability
   `gbest_prob` the partner is the global-best set, otherwise a random source.
   A candidate replaces its parent if it improves `sigma` beyond the Monte-Carlo
   tolerance, or if it is tied on `sigma` and improves `Phi`.
2. **Onlooker bees.** Sources are drawn by tournament and make the same move;
   better sources therefore receive more evaluations without collapsing onto a
   single answer.
3. **Scouts.** A source that has failed `limit` times is rebuilt near the
   incumbent (70 %) or at random (30 %).
4. **Dedupe.** Exact duplicate sets are replaced by fresh biased samples.
5. **Elitism.** The best set ever seen is retained, which is what makes the
   CELF warm start a lower bound on the final answer.

The budget is checked before every move, so a run can never exceed
`k * |P|` evaluations. The refinement phase of the production code (W6, W8, W9)
is omitted here for readability; it is disabled in every reported run
(`refine_every=0`).

In [7]:
def abc_optimize(model, k, pool, cfg):
    """One ABC restart. Returns (seeds, fitness, trace, evals, seconds)."""
    rng = np.random.default_rng(cfg.random_state)
    t0, e0 = time.perf_counter(), model.n_evals
    pool = list(pool)
    n_sources = max(4, cfg.colony_size // 2)

    def spent():
        return model.n_evals - e0

    def budget_left():
        return cfg.eval_budget is None or spent() < cfg.eval_budget

    # ---- initialisation: CELF warm start plus biased random sets ----
    population = []
    if cfg.seed_with_celf:
        population.append(celf(model, k, pool))
    while len(population) < n_sources:
        population.append(_biased_set(model, pool, k, rng))
    population = population[:n_sources]

    def fitness_of(s):
        sigma, phi, sem = model.sigma_potential(s)
        return sigma, sigma + cfg.potential_weight * phi, sem

    fit_pairs = [fitness_of(s) for s in population]
    fitness = [f for _, f, _ in fit_pairs]
    trials = [0] * n_sources
    best_i = int(np.argmax([s for s, _, _ in fit_pairs]))
    best_set, best_fit = list(population[best_i]), fit_pairs[best_i][0]
    trace = []

    def accept(new_sigma, new_phi, new_sem, i):
        """Lexicographic (sigma, Phi) acceptance with noise tolerance."""
        cur_sigma, cur_phi, cur_sem = fit_pairs[i]
        tol = cfg.tol_mult * max(cur_sem, new_sem)
        if new_sigma > cur_sigma + tol:
            return True
        if abs(new_sigma - cur_sigma) <= tol:
            return new_phi > cur_phi + 1e-9 or (cfg.accept_equal
                                                and abs(new_phi - cur_phi) <= 1e-9)
        return False

    def dedupe():
        seen = set()
        for idx in range(n_sources):
            key = frozenset(population[idx])
            if key in seen:
                population[idx] = _biased_set(model, pool, k, rng)
                fit_pairs[idx] = fitness_of(population[idx])
                fitness[idx] = fit_pairs[idx][1]
                trials[idx] = 0
            seen.add(key)

    # ---- generations ----
    for cycle in range(1, cfg.max_cycles + 1):
        if not budget_left():
            break

        # employed bees
        for i in range(n_sources):
            if not budget_left():
                break
            partner = (best_set if rng.random() < cfg.gbest_prob
                       else population[rng.integers(n_sources)])
            cand = _candidate(population[i], partner, pool, k, rng, cfg)
            ns, nph, nse = fitness_of(cand)
            if accept(ns, nph, nse, i):
                population[i], trials[i] = cand, 0
                fit_pairs[i] = (ns, nph, nse)
                fitness[i] = ns + cfg.potential_weight * nph
            else:
                trials[i] += 1

        # onlooker bees
        for i in _pick_sources(fitness, rng, cfg):
            if not budget_left():
                break
            partner = (best_set if rng.random() < cfg.gbest_prob
                       else population[rng.integers(n_sources)])
            cand = _candidate(population[i], partner, pool, k, rng, cfg)
            ns, nph, nse = fitness_of(cand)
            if accept(ns, nph, nse, i):
                population[i], trials[i] = cand, 0
                fit_pairs[i] = (ns, nph, nse)
                fitness[i] = ns + cfg.potential_weight * nph
            else:
                trials[i] += 1

        # scouts
        for i in range(n_sources):
            if trials[i] > cfg.limit and budget_left():
                if cfg.directed_scout and rng.random() < 0.7:
                    cand = _neighbour_solution(
                        best_set, best_set, pool, k, rng,
                        ABCConfig(n_mod=2, mutation_rate=1.0,
                                  gbest_prob=0.0))
                else:
                    cand = rng.choice(pool, size=k,
                                      replace=False).tolist()
                population[i] = cand
                fit_pairs[i] = fitness_of(cand)
                fitness[i] = fit_pairs[i][1]
                trials[i] = 0

        # duplicate suppression and elitism
        if cfg.dedupe:
            dedupe()
        i = int(np.argmax([s for s, _, _ in fit_pairs]))
        if fit_pairs[i][0] > best_fit:
            best_fit, best_set = fit_pairs[i][0], list(population[i])
        trace.append(best_fit)

    return best_set, best_fit, trace, spent(), time.perf_counter() - t0

## 7. Demo on the nine-node graph

Two seeds, a colony of eight sources, and a 200-evaluation budget. On this
graph every singleton seed leaves the cascade at the seed itself, so a
marginal-gain method sees zero gain everywhere. Pairs such as $\{3,6\}$ and
$\{7,8\}$ flip six nodes; the colony should find one of them.

In [8]:
cfg = ABCConfig(colony_size=8, max_cycles=30, eval_budget=200,
                random_state=1)
seeds, fit, trace, evals, seconds = abc_optimize(
    toy, 2, list(range(1, N + 1)), cfg)

state, _ = toy._run(seeds)
print(f"seeds      : {sorted(seeds)}")
print(f"fitness    : {fit:.1f}")
print(f"evaluations: {evals}")
print(f"time       : {seconds:.3f} s")
print(f"final state: {state[1:].tolist()}")
print(f"positive   : {int((state == 1).sum())}   negative: {int((state == -1).sum())}")

seeds      : [7, 8]
fitness    : 6.0
evaluations: 45
time       : 0.008 s
final state: [-1, 0, 1, 1, 1, 1, 1, 1, 0]
positive   : 6   negative: 1


In [9]:
# Context: the best seed pairs on this graph (net spread, positive, negative)
import itertools

rows = []
for S in itertools.combinations(range(1, N + 1), 2):
    st, _ = toy._run(S)
    pos, neg = int((st == 1).sum()), int((st == -1).sum())
    rows.append((pos - neg, pos, neg, S))
rows.sort(reverse=True)
for net, pos, neg, S in rows[:5]:
    print(f"{S}: net={net}  positive={pos}  negative={neg}")

(7, 8): net=5  positive=6  negative=1
(3, 6): net=5  positive=6  negative=1
(4, 8): net=4  positive=4  negative=0
(3, 8): net=4  positive=4  negative=0
(3, 5): net=3  positive=3  negative=0


## 8. The same code on Bitcoin Alpha

Nothing changes except the model. The production model evaluates 30 threshold
draws per call, so the spread estimates are averages rather than a single
cascade. The run below uses the exact settings of the report: colony 30,
80 cycles, budget $10\times200=2000$ evaluations, restart seed 600.

In [10]:
from graphio import load_graph
from metrics import get_metrics, signed_pool
from opinion_model import SignedOpinionModel

G = load_graph("bitcoin_alpha", verbose=False)
met = get_metrics(G)

model = SignedOpinionModel(G, 30, seed=7, comm=met["comm"])
pool = signed_pool(model, met["pr"], met["core"], 200, seed=11)

cfg = ABCConfig(colony_size=30, max_cycles=80, eval_budget=10 * 200,
                random_state=600)
btc_seeds, btc_fit, btc_trace, btc_evals, btc_seconds = abc_optimize(
    model, 10, pool, cfg)

held = SignedOpinionModel(G, 300, seed=999_999, comm=met["comm"])
pos, neg, net, div, fair = held.components(btc_seeds)

print(f"seeds       : {sorted(btc_seeds)}")
print(f"search fit  : {btc_fit:.1f} positive nodes (30 draws)")
print(f"held-out    : pos={pos:.1f}  neg={neg:.1f}  net={net:.1f}")
print(f"evaluations : {btc_evals}")
print(f"time        : {btc_seconds:.1f} s")

[backend] GPU (torch-cuda)=OFF -> numpy CPU


seeds       : [0, 1, 2, 3, 5, 6, 12, 15, 128, 176]
search fit  : 550.1 positive nodes (30 draws)
held-out    : pos=551.2  neg=7.8  net=543.3
evaluations : 2001
time        : 12.7 s


## 9. Equivalence with the project module

The notebook code is a re-typed, commented version of
`signed_bench/abc_proposed.py`. To show that the two are behaviourally
identical, both are run on the toy graph with the same configuration and seed.
They must return the same seed set and the same fitness.

In [11]:
from abc_proposed import ABCConfig as ModuleConfig
from abc_proposed import abc_optimize as module_abc

# run the notebook implementation again on a fresh copy of the toy model
toy_a = ToyOpinionModel(EDGES, N)
cfg_toy = ABCConfig(colony_size=8, max_cycles=30, eval_budget=200,
                    random_state=1)
nb_seeds, nb_fit, _, _, _ = abc_optimize(toy_a, 2, list(range(1, N + 1)),
                                         cfg_toy)

# run the project module on another fresh copy with the same seed
toy_b = ToyOpinionModel(EDGES, N)
module_result = module_abc(
    toy_b, 2, list(range(1, N + 1)),
    ModuleConfig(colony_size=8, max_cycles=30, eval_budget=200,
                 random_state=1))

print("module seeds :", sorted(module_result.seeds),
      "fitness", module_result.spread)
print("notebook     :", sorted(nb_seeds), "fitness", nb_fit)
assert sorted(module_result.seeds) == sorted(nb_seeds)
assert abs(module_result.spread - nb_fit) < 1e-9
print("\nmatch: OK")

module seeds : [7, 8] fitness 6.0
notebook     : [7, 8] fitness 6.0

match: OK


## 10. Where things live

| file | contents |
|---|---|
| `signed_bench/abc_proposed.py` | canonical implementation of this notebook |
| `signed_bench/opinion_model.py` | two-opinion diffusion model (numpy / torch) |
| `signed_bench/run_opinion.py` | benchmark runner (resumable CSV) |
| `signed_bench/results_opinion.csv` | 117 measured cells |
| `signed_bench/OPINION_RESULTS.md` | tables and discussion |
| `REPORT.pdf` | model, proofs, full results |

**Parameter summary (reported runs):** colony 30 (15 sources), 80 cycles,
abandonment limit 6, gbest probability 0.6, tournament 4, one guided swap plus
a 0.2 random swap, exploration 0.15, plateau weight 0.05, acceptance tolerance
2 standard errors, budget $k\times200$, refinement off, restarts 3 (ABC) and
5 (ABCVNS), search draws 30, held-out draws 300.